<a href="https://colab.research.google.com/github/jun-1993-p/rag_example_jun.1993.9/blob/main/7%EA%B0%95_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 사전 작업.
  1. 런타임 유형 진입.
  2. T4 GPU 선택.

In [ ]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time

proc = subprocess.Popen(["ollama", "serve"],
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL)
time.sleep(5)

In [ ]:
!ollama pull gemma4:12b

In [ ]:
import ollama

response = ollama.chat(
    model = "gemma4:12b",
    messages = [{"role" : "user", "content" : "안녕, 잘 지내?"}]
)

print(response["message"]["content"])

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

# dwg to dxf

: 오픈 포맷으로 변경하는 작업.

In [ ]:
import shutil, subprocess, requests

if shutil.which("dwg2dxf") is None:
    # apt에 있으면 그걸로 설치
    subprocess.run("apt-get install -y libredwg-tools", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if shutil.which("dwg2dxf") is None:
    # 없으면 최신 릴리스를 받아 소스 빌드
    rel = requests.get("https://api.github.com/repos/LibreDWG/libredwg/releases/latest").json()
    url = next(a["browser_download_url"] for a in rel["assets"] if a["name"].endswith(".tar.xz"))
    print("빌드:", url)
    cmd = f"""
    cd /tmp && wget -q {url} -O libredwg.tar.xz && mkdir -p libredwg &&
    tar xf libredwg.tar.xz -C libredwg --strip-components=1 && cd libredwg &&
    ./configure --disable-bindings > /tmp/build.log 2>&1 &&
    make -j$(nproc) >> /tmp/build.log 2>&1 &&
    make install >> /tmp/build.log 2>&1 && ldconfig
    """
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        print(open("/tmp/build.log").read()[-3000:])

print(subprocess.run("dwg2dxf --version", shell=True, capture_output=True, text=True).stdout)

빌드: https://github.com/LibreDWG/libredwg/releases/download/0.14/libredwg-0.14.tar.xz
dwg2dxf 0.14



In [ ]:
!pip install -q ezdxf matplotlib ollama

import os, shutil, subprocess
from google.colab import files

uploaded = files.upload()
src = next(iter(uploaded))

# 파일명에 한글·괄호·공백이 있어서 단순한 이름으로 복사
shutil.copy(src, "input.dwg")
r = subprocess.run(["dwg2dxf", "-y", "-o", "input.dxf", "input.dwg"],
                   capture_output=True, text=True)
print(r.stderr[-1500:])   # 경고는 흔하니 파일이 생겼는지가 중요
print("DXF 생성:", os.path.exists("input.dxf"), os.path.getsize("input.dxf") if os.path.exists("input.dxf") else 0)

Saving 농림-22-37-가_건축물대장용_배치도(120).dwg to 농림-22-37-가_건축물대장용_배치도(120).dwg
to short handle stream

DXF 생성: True 4892712


In [ ]:
import ezdxf
from ezdxf import bbox, recover
from ezdxf.addons.drawing import matplotlib as dxf_plot
from collections import Counter

def summarize_dxf(path, max_items=60):
    try:
        doc = ezdxf.readfile(path)
    except ezdxf.DXFStructureError:
        doc, _ = recover.readfile(path)
    msp = doc.modelspace()
    out = []

    out.append(f"[원본 파일명] {src}")
    out.append(f"[CAD 버전] {doc.dxfversion} ({doc.acad_release})")
    out.append(f"[단위 코드 $INSUNITS] {doc.header.get('$INSUNITS', 0)}")
    out.append("[레이아웃] " + ", ".join(doc.layouts.names()))

    ext = bbox.extents(msp)
    if ext.has_data:
        out.append(f"[도면 범위] 가로 {ext.size.x:.0f} x 세로 {ext.size.y:.0f}")

    layer_cnt = Counter(e.dxf.layer for e in msp)
    out.append(f"[레이어 {len(doc.layers)}개]")
    for layer in doc.layers:
        n = layer.dxf.name
        out.append(f"  - {n}: 색상 {layer.dxf.color}, 선종류 {layer.dxf.linetype}, 객체 {layer_cnt.get(n, 0)}개")

    out.append("[객체 종류별 개수] " + ", ".join(f"{k} {v}" for k, v in Counter(e.dxftype() for e in msp).most_common()))

    blocks = Counter(e.dxf.name for e in msp.query("INSERT"))
    if blocks:
        out.append("[블록 삽입] " + ", ".join(f"{k} {v}" for k, v in blocks.most_common(20)))

    texts = [e.dxf.text for e in msp.query("TEXT")] + [e.plain_text() for e in msp.query("MTEXT")]
    out.append("[문자] " + " | ".join(t.strip() for t in texts[:max_items] if t.strip()))

    dims = []
    for d in msp.query("DIMENSION"):
        try:
            v = d.get_measurement()
            dims.append(f"{v:.0f}" if isinstance(v, (int, float)) else str(v))
        except Exception:
            pass
    out.append(f"[치수값 {len(dims)}개] " + " | ".join(dims[:max_items]))

    dxf_plot.qsave(msp, "plan.png", bg="#FFFFFF", dpi=150)
    return "\n".join(out)

info = summarize_dxf("input.dxf")
print(info)

[원본 파일명] 농림-22-37-가_건축물대장용_배치도(120).dwg
[CAD 버전] AC1015 (R2000)
[단위 코드 $INSUNITS] 0
[레이아웃] Layout1, Model
[도면 범위] 가로 50397 x 세로 34389
[레이어 28개]
  - 0: 색상 7, 선종류 Continuous, 객체 0개
  - 2D 제도 - 중심선.2D_펜_번호__1: 색상 1, 선종류 Continuous, 객체 0개
  - 도면 _ 텍스트(공통)_펜_번호__3: 색상 3, 선종류 Continuous, 객체 0개
  - 0_펜_번호__3: 색상 3, 선종류 Continuous, 객체 3개
  - W-18_배치도(2층형B)_1_신규_대지 _ 계획대지_펜_번호__15: 색상 7, 선종류 Continuous, 객체 3개
  - W-18_배치도(2층형B)_1_신규_대지 _ 계획대지_펜_번호__8: 색상 8, 선종류 Continuous, 객체 1개
  - W-18_배치도(2층형B)_1_신규_건축 _ 슬래브_펜_번호__5: 색상 5, 선종류 Continuous, 객체 7개
  - W-18_배치도(2층형B)_1_신규_건축 _ 슬래브_펜_번호__3: 색상 3, 선종류 Continuous, 객체 7개
  - W-18_배치도(2층형B)_1_신규_건축 _ 슬래브_펜_번호__4: 색상 4, 선종류 Continuous, 객체 22개
  - W-18_배치도(2층형B)_1_신규_대지 _ 식재_펜_번호__4: 색상 4, 선종류 Continuous, 객체 14개
  - W-18_배치도(2층형B)_1_신규_대지 _ 계획대지_펜_번호__5: 색상 5, 선종류 Continuous, 객체 6960개
  - W-18_배치도(2층형B)_1_신규_대지 _ 식재_펜_번호__5: 색상 5, 선종류 Continuous, 객체 6개
  - W-18_배치도(2층형B)_1_신규_공통 _ 그리드(공통)_펜_번호__4: 색상 4, 선종류 Continuous, 객체 24개
  - W-18_배치도(2층형B)_1_신규_공통

In [ ]:
import ollama

prompt = f"""다음은 CAD 도면(DWG를 DXF로 변환)에서 추출한 정보이고, 첨부 이미지는 해당 도면을 렌더링한 것입니다.
이 도면을 한국어 마크다운으로 분석해 주세요.

### 1. 도면 개요
### 2. 도면 구성 (단면·상세 부위)
### 3. 치수 정보
### 4. 도면의 기술적 정보 (CAD 버전, 레이어 구성과 각 레이어의 역할 등)
### 5. 도형 및 기호

추출 정보에 없는 내용은 추측이라고 밝혀 주세요.

[추출 정보]
{info}

마지막 단계로 이 정보를 보기 좋게 표로 정리해주세요.
"""

response = ollama.chat(
    model="gemma4:12b",
    messages=[{"role": "user", "content": prompt, "images": ["plan.png"]}],
    options={"num_ctx": 16384},
)
print(response["message"]["content"])

제공된 CAD 도면 정보와 렌더링 이미지를 바탕으로 분석한 결과입니다.

---

# 📋 건축 도면 분석 보고서

### 1. 도면 개요
*   **도면명:** 농림-22-37-가_건축물대장용_배치도(120).dwg
*   **도면 종류:** 배치도 (Site Plan)
*   **사용 목적:** 건축물대장 등록용 (대지 내 건축물의 배치 및 위치 관계를 나타냄)
*   **표기 축척:** 1:120
*   **도면 번호:** A-004

### 2. 도면 구성 (단면·상세 부위)
도면은 대지 내 건축물의 배치와 주변 시설과의 관계를 시각화하고 있습니다.
*   **건축물 영역:** 
    *   **지상 1층:** 해치(Hatch) 패턴 1번으로 표시된 영역.
    *   **지상 2층:** 해치(Hatch) 패턴 2번으로 표시된 영역.
    *   **발코니:** 건축물 외곽에 인접한 공간으로 구분됨.
*   **부지 내 시설:**
    *   **마당:** 건축물 외곽의 여유 공간.
    *   **주차장:** 2대의 차량이 주차 가능한 공간으로 표시됨.
    *   **옥외수전:** 선택적으로 적용되는 시설로 명시됨.
*   **인프라 및 경계:**
    *   **인접대지경계선:** 필지 경계를 명확히 구분.
    *   **도로:** 4M 이상의 현황도로와 인접.
    *   **상하수도 연결:** 
        *   **오수관:** D-100 (VG1 PVC관 K.S제품) 및 D-200 (PE이중벽관 K.S제품)로 시 오수관 본관에 연결.
        *   **상수도:** 시 상수관 본관에 연결되는 지점 표시.
    *   **배수:** 부지 내 우수관 연결(기존구거) 정보 포함.

### 3. 치수 정보
도면 내에 표기된 주요 수치 정보입니다. (단위는 도면 설정에 따름)
*   **주요 좌표/치수:** 2,199 | 9,600 | 3,801 | 6,672 | 9,900 | 2,003
*   **관로 규격:** 
    